# Lab 09 · Evaluation: a multi-agent system against a single model

**Prepared by Mustafa Assaf** · Self-study: Go deeper · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Warm-up · Evaluate answers against known answers

Before comparing systems, define the same questions and expected answers for each. These are made-up answers used to demonstrate scoring; they are not model performance measurements.

In [ ]:
questions = ['What is 2 + 2?', 'What colour is a clear sky?']
print('Questions:', questions)
expected = ["4", "blue"]
sample_a = ["4", "blue"]
sample_b = ["5", "blue"]
correct_a, correct_b = 0, 0
for index in range(len(expected)):
    if sample_a[index] == expected[index]:
        correct_a += 1
    if sample_b[index] == expected[index]:
        correct_b += 1
print("Sample A correct:", correct_a, "/ 2")
print("Sample B correct:", correct_b, "/ 2")

**Next:** Apply the same comparison to architecture answers, then consider evidence, cost, and judge calibration.

## Evaluate recommendations against architecture scenarios
**Start with a problem:** A council’s convincing answer does not show whether it is better than a single model.

**In this lab:** Compare both on the same cases using explicit checks. This lab’s answers and timings are teaching fixtures, not measurements.

**New:** compare single-model and council answers on the *same* labelled cases. In these examples, broad support search suggests CQRS; a small order-details lookup by ID suggests API composition.

The table below contains **invented teaching fixtures**, including call counts and timings. They were not measured from Qwen, DeepSeek, Gemini, or a council, and must not be reported as model benchmarks. Only the judge calls a model. For a real experiment, collect fresh answers, actual wall-clock time, token usage, and human ratings from every system.

We distinguish label agreement, valid citation IDs, and a stated downside. A valid ID does not prove that a source supports the claim.

**At a glance**

![Lab 09 at a glance: Synthetic run fixtures (4 systems × 2 cases), then Correct · citation IDs (calls · seconds), then LLM judge (names a cost?), then Calibrate (agree with human ≥ 75%), then Council vs best (wins, and at what cost).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/09.svg)

## Think about the example

A small order-details lookup needs data for one ID; a support search must filter across many records. API composition handles small joins; CQRS adds a searchable read model. Event sourcing keeps an event history that can be replayed to recover past state.

**Concepts:** Choose data patterns from requirements; Measure label agreement, citation validity, and trade-offs.

Use the scenario above to consider these questions before running the code:

1. For a small order-details lookup by ID, why might a separate CQRS read model add unnecessary work?
2. Which data pattern supports replaying a historical account balance?

## The idea in context

Evaluation needs consistent inputs, labelled criteria, and honest provenance. Synthetic answers and timings teach the method without claiming measured model performance.

Use actual run logs and repeated scenarios for the project experiment.

| Key term | Meaning |
| --- | --- |
| **Baseline** | The system to beat; here, the best single model. |
| **Metric** | A number that measures one quality, such as correct answers or model calls. |
| **LLM judge** | A model that rates other models' answers against a rubric. |
| **Calibration** | Checking the judge against human ratings before trusting it. |

## Pattern · Evaluating a pattern

This lab is not a pattern of its own. It measures the quality attributes that the pattern cards in Labs 06–08 name, such as Accuracy, Cost, and Latency, for *Voting-Based Cooperation* and *Debate-Based Cooperation*. The LLM judge is one of the aggregation options that the field guide lists for Voting-Based Cooperation; here it rates answers, and it is trusted only after calibration.

## Main exercise at a glance

1. Two scenarios and synthetic answers
2. Compute explicit measures
3. Ask a judge one narrow question
4. Calibrate against human ratings
5. Compare with the best single-model fixture

## Step 1 · Two scenarios and synthetic answers

The expected labels are introductory choices for these two cases, not unique answers to every real architecture problem. Each tuple holds the option, citations, explanation, illustrative calls, and illustrative seconds.

Each `runs` entry is a five-item tuple: `(option, citations, answer, calls, seconds)`. The key is `(system, scenario)`, so every answer can be compared against the same scenario.

In [ ]:
import json

SOURCES = {"R2", "R3", "R6"}
expert = {"Support search": "CQRS", "Order details": "API composition"}
runs = {  # SYNTHETIC fixtures, not observed provider outputs or measured timings
    ("Qwen", "Support search"): ("API composition", ["R2"], "One composer is simple.", 1, 1.6),
    ("Qwen", "Order details"): ("API composition", ["R2"], "Lookup by ID fits a composer.", 1, 1.5),
    ("DeepSeek", "Support search"): ("CQRS", ["R3"], "Rich search is fast, but the view may lag.", 1, 2.2),
    ("DeepSeek", "Order details"): ("API composition", [], "A composer is easy.", 1, 2.0),
    ("Gemini", "Support search"): ("API composition", [], "A composer is always best.", 1, 1.9),
    ("Gemini", "Order details"): ("API composition", ["R2"], "Small lookup; a source outage can fail it.", 1, 2.1),
    ("Council", "Support search"): ("CQRS", ["R3", "R6"], "Rich search, at the cost of a stale extra view.", 5, 9.8),
    ("Council", "Order details"): ("API composition", ["R2"], "Small lookup; all source services must respond.", 5, 9.1),
}
systems = ["Qwen", "DeepSeek", "Gemini", "Council"]

## Step 2 · Compute explicit measures
The loops visit each system and each scenario. Count matching labels and valid citations, then add the illustrative calls and seconds. The numbers demonstrate accounting; they are not actual provider measurements.

In [ ]:
scores = {}
for system in systems:
    result = {"correct": 0, "citation_valid": 0, "calls": 0, "seconds": 0}
    for scenario in expert:
        if (system, scenario) not in runs:
            raise ValueError("Every system must answer every scenario.")
        option, citations, answer, calls, seconds = runs[(system, scenario)]
        if option == expert[scenario]:
            result["correct"] += 1
        if citations and set(citations) <= SOURCES:
            result["citation_valid"] += 1
        result["calls"] += calls
        result["seconds"] += seconds
    result["seconds"] = round(result["seconds"], 1)
    scores[system] = result
    print("SYNTHETIC", system, result)

## Step 3 · Ask a judge one narrow question
For each example answer, ask a model whether it names a downside. The loop sends and reads each request directly. A judge can be wrong; Step 4 compares it with human ratings.

In [ ]:
RUBRIC = ("Does this answer name a cost or downside of the option it chose? "
          'Reply with JSON only, no code fences: {"names_cost": true or false}')
judged = {}
for key, run in runs.items():
    option, citations, answer, calls, seconds = run
    reply = ask(f"{RUBRIC}\nOption: {option}\nAnswer: {answer}")
    judged[key] = json.loads(reply)["names_cost"] is True
    print(key, "names a cost:", judged[key])

## Step 4 · Calibrate against human ratings

A four-answer sample only demonstrates calibration. The example threshold accepts at least 75% agreement. A real evaluation needs a representative calibration set, error analysis, and a separate rating set.

In [ ]:
human = {("Qwen", "Support search"): False, ("DeepSeek", "Support search"): True,
         ("DeepSeek", "Order details"): False, ("Gemini", "Order details"): True}
agree = 0
for key, rating in human.items():
    if judged[key] == rating:
        agree += 1
    else:
        print("Disagreement:", key, runs[key][2], "judge:", judged[key], "human:", rating)
trusted = agree / len(human) >= 0.75
print(f"Judge-human agreement: {agree}/{len(human)}. Use judge metric: {trusted}")

## Step 5 · Compare with the best single-model fixture

Select the strongest baseline by label agreement and then valid citations. An untrusted judge metric remains `None`. This prints a comparison of fixtures, not a claim that any named model or council won an experiment.

The baseline is chosen by correct answers first and valid citation IDs second. These two illustrative cases are too few to establish a real ranking.

In [ ]:
for system in systems:
    count = 0
    for (name, scenario), verdict in judged.items():
        if name == system and verdict:
            count += 1
    scores[system]["names_cost"] = count if trusted else None
best = systems[0]
for system in systems[1:3]:
    current = (scores[system]["correct"], scores[system]["citation_valid"])
    previous = (scores[best]["correct"], scores[best]["citation_valid"])
    if current > previous:
        best = system
for metric in ("correct", "citation_valid", "names_cost", "calls", "seconds"):
    print(f"{metric:<15} council {scores['Council'][metric]!s:<6} {best} fixture {scores[best][metric]}")
wins = []
for metric in ("correct", "citation_valid", "names_cost"):
    if scores["Council"][metric] is not None and scores["Council"][metric] > scores[best][metric]:
        wins.append(metric)
ratio = scores["Council"]["calls"] / scores[best]["calls"]
print("Fixture comparison: council wins on", wins, "with", ratio, "times the illustrative calls.")

## What you should see

The DeepSeek and council fixtures each match 2/2 labels; Qwen and Gemini match 1/2. Judge agreement and the stated-cost metric depend on the model replies. The council fixture has more valid citations and five times the illustrative calls. None of these numbers are empirical model results.

## Try it

1. Change the required judge-human agreement to 80%. When is `names_cost` reported as `None`?
2. Give a false explanation a known citation ID. What extra check is needed beyond checking whether the citation ID exists?
3. Add two cases with explicit human labels: a trip booking that must compensate earlier reservations when payment fails, and a bank that needs historical balances and an audit trail.
4. Collect one real answer from each system on each scenario. Replace synthetic timings with measured wall time and report token usage separately from elapsed time.

## Takeaway

Separate fixture-based plumbing tests from empirical evaluation. Compare identical scenarios, calibrate the judge, and report trade-offs and cost. The microservices revision matters too: CQRS, API composition, sagas, and event sourcing solve different problems.

## Optional extension · Save the synthetic comparison with MLflow

After the core steps, run `%pip install -q mlflow==2.20.0` in a new cell, then this extra cell. This logs the **synthetic fixture** metrics, not new provider benchmarks. Missing/untrusted judge metrics are omitted, never converted to zero. Tracking is optional and is not part of the five core steps.

```python
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("synthetic-microservice-pattern-demo")
for system, metrics in scores.items():
    with mlflow.start_run(run_name=system):
        mlflow.log_params({"system": system, "data_kind": "synthetic teaching fixture"})
        recorded = {}
        for name, value in metrics.items():
            if value is not None:
                recorded[name] = value
        mlflow.log_metrics(recorded)
```

In Colab, the tracking database `mlflow.db` lives in the runtime filesystem; download it before ending the session. On your own computer, use `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000` and open localhost:5000 there. A Colab runtime's localhost is not your laptop's localhost.